## Phase — Data Quality

## Objective

Assess the quality, consistency, validity, and reliability of the processed loan dataset before using it for business analysis and machine learning.

## Key Questions

- Are there missing values that require treatment?
- Are numerical values within reasonable ranges?
- Are there suspicious or invalid values?
- Are categorical values consistent?
- Are there significant outliers?
- Which data quality issues require correction?
- Does the final dataset remain suitable for downstream analysis and modeling?



### 4.1 Load Processed Dataset

The processed dataset generated during Phase 3 will be loaded for data quality assessment.

The raw dataset will not be modified. All quality checks and subsequent cleaning decisions will be performed on the processed dataset.

In [3]:
import pandas as pd

df = pd.read_csv("../data/processed/LC_loans_granting_model_dataset_processed.csv")

print("Dataset Shape: ",df.shape)

C:\Users\jeet\AppData\Local\Temp\ipykernel_15432\1115886580.py:3: DtypeWarning: Columns (14: desc) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/LC_loans_granting_model_dataset_processed.csv")


Dataset Shape:  (1347681, 15)


### 4.2 Missing Value Analysis

Missing values can affect statistical analysis, business reporting, and machine learning.

We will identify:
- Which columns contain missing values
- How many values are missing
- What percentage of each column is missing

No missing values will be removed or imputed at this stage.

In [8]:
missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (df.isnull().sum() / len(df)) *100
})

missing_summary = missing_summary[
    missing_summary['Missing Count'] > 0
].sort_values(
    by='Missing Count',
    ascending=False
)

missing_summary

,Missing Count,Missing Percentage
desc,1228582,91.162671
title,16657,1.235975
zip_code,1,0.000074


### 4.2.1 Missing Value Interpretation

The `desc` column has approximately 91% missing values, making it unsuitable for direct use without further consideration.

The `title` column has approximately 1.24% missing values, which is relatively small but should still be evaluated before deciding on treatment.

The `zip_code` column has only one missing value, representing a negligible proportion of the dataset.

No missing values will be removed or imputed yet. Each column will be evaluated based on its business meaning and usefulness for downstream analysis.

### 4.3 Investigation of `desc`

The `desc` column contains borrower-provided loan descriptions.

Because approximately 91% of the values are missing, we need to determine whether the available descriptions provide enough consistent information to justify keeping this feature for downstream analysis.

The column will not be filled or modified until its usefulness has been evaluated.

In [11]:
df['desc'].dropna().head(10)

1296      We knew that using our credit cards to finance...
29168     I had a bad year two years ago, with some late...
33329     Lenders, I have the ability to pay off my curr...
44444     I paid off my first Prosper loan, but had an e...
59975     I want to cut down on my credit card debt now ...
64278     i recently proposed to my girlfriend of almost...
107756    I would like to consolidate my loan in one sma...
145800    I have excellent credit and I need to reduce m...
163023    I have been in business for a year and want to...
217175    I am building a new garage on a property that ...
Name: desc, dtype: str

### 4.3.1 Decision — `desc`

The `desc` column contains borrower-provided free-text loan descriptions.

Although the available values contain potentially useful information, approximately 91.16% of the column is missing.

Because this project focuses on interpretable credit-risk analysis and structured borrower and loan characteristics, the `desc` column will be excluded from the main analytical dataset.

No imputation will be performed because replacing such a large proportion of missing free-text data would not provide meaningful information.

The original raw and processed datasets remain unchanged.

### 4.4 Investigation of `title`

The `title` column contains the borrower-provided title of the loan.

Approximately 1.24% of the values are missing, which is a relatively small proportion of the dataset.

Before deciding how to handle the missing values, the available values will be inspected to understand the nature and usefulness of this feature.

In [13]:
df['title'].dropna().head(10)

0          Debt consolidation
1                    Business
3              Major purchase
4          Debt consolidation
5          Debt consolidation
6              Major purchase
7     Credit card refinancing
8     Credit card refinancing
10         Debt consolidation
11         Debt consolidation
Name: title, dtype: str

### 4.4.1 Decision — `title`

The `title` column contains meaningful borrower-provided loan titles such as debt consolidation, business, major purchase, and credit card refinancing.

Only approximately 1.24% of the values are missing.

Because the missing proportion is small and the feature may contain useful information, the `title` column will be retained.

Missing values will be represented as `Unknown` rather than removing the corresponding loan records or imputing an artificial title.

The actual transformation will be applied later after all data-quality decisions have been reviewed.

### 4.5 Investigation of `zip_code`

The `zip_code` column contains the borrower's ZIP code.

Only one value is missing, representing an extremely small proportion of the dataset.

The impact of this missing value will be evaluated before deciding how to handle it.

In [15]:
df[df['zip_code'].isna()]

,id,issue_d,revenue,dti_n,loan_amnt,fico_n,experience_c,emp_length,purpose,home_ownership_n,addr_state,zip_code,Default,title,desc
999330,96387212,2017-01-01,40000.0,22.86,5600,672.0,1,10+ years,debt_consolidation,MORTGAGE,AL,NaN,0,Debt consolidation,NaN


### 4.5.1 Decision — `zip_code`

The `zip_code` column contains only one missing value, representing approximately 0.00007% of the dataset.

The corresponding loan record contains other valid borrower and loan information.

The row will be retained because removing it would unnecessarily discard a valid loan record.

The missing ZIP code will be represented as `Unknown` rather than being inferred from the borrower's state.

The actual transformation will be applied later after all data-quality decisions have been reviewed.

### 4.6 Investigation of Suspicious `dti_n` Values

During Phase 2, unusually high values were identified in the `dti_n` column, including `999`.

A debt-to-income ratio of 999 appears unusually high and may represent a special or invalid value rather than a genuine ratio.

Before modifying these records, the value will be investigated to determine its frequency and relationship with the target variable.

No values will be changed at this stage.

In [16]:
df[df['dti_n']==999]['Default'].value_counts()

Default
0    29
1     9
Name: count, dtype: int64

In [17]:
df[df['dti_n'] > 100]['dti_n'].describe()

count    533.000000
mean     266.213583
std      250.901005
min      100.040000
25%      119.590000
50%      161.940000
75%      265.600000
max      999.000000
Name: dti_n, dtype: float64

In [18]:
df[df['dti_n'] >= 999]['dti_n'].value_counts().sort_values()

dti_n
999.0    38
Name: count, dtype: int64

## 4.6.1 Decision — `dti_n = 999`

The `dti_n` column contains 533 observations above 100.

Among these, 38 observations have values greater than or equal to 900, and all 38 are exactly 999.

Because 999 appears as a distinct extreme value rather than as part of a continuous range of high DTI values, it is treated as a special/placeholder value.

The 38 observations with `dti_n = 999` will be converted to missing (`NaN`) rather than removing the corresponding loan records.

Other high `dti_n` values above 100 will be retained for further outlier analysis and will not be removed solely because they are large.

## 4.7 Numerical Outlier Analysis

Extreme numerical values can affect statistical analysis and machine-learning models.

The numerical features will be examined individually to determine whether extreme observations are valid values, potential data-entry issues, or statistical outliers.

We will begin with `revenue`.

In [19]:
df['revenue'].describe()

count    1.347681e+06
mean     7.736968e+04
std      7.036299e+04
min      1.896000e+03
25%      4.660000e+04
50%      6.500000e+04
75%      9.200000e+04
max      1.099920e+07
Name: revenue, dtype: float64

### 4.7.1 Revenue Outlier Detection Using IQR

The `revenue` distribution has a large difference between the median and maximum value.

The Interquartile Range (IQR) method will be used to identify statistically extreme observations.

The IQR method identifies observations outside the typical middle 50% of the data.

This step identifies potential outliers; it does not automatically mean that those records should be removed.

In [20]:
Q1 = df['revenue'].quantile(0.25)
Q3 = df['revenue'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print('Q1: ',Q1)
print("Q3: ",Q3)
print("IQR: ",IQR)
print("Lower Bound: ",lower_bound)
print("Upper Bound: ",upper_bound)

Q1:  46600.0
Q3:  92000.0
IQR:  45400.0
Lower Bound:  -21500.0
Upper Bound:  160100.0


### 4.7.2 Revenue Outlier Count

Using the IQR method, revenue values above 160,100 are statistically classified as potential upper outliers.

The number of affected records will be calculated to assess their proportion of the dataset before deciding on treatment.

In [21]:
revenue_outliers = df[df['revenue']>upper_bound]

print("Revenue outliers Count: ",len(revenue_outliers))
print("revenue outliers Per: ",(len(revenue_outliers)/len(df)*100))

Revenue outliers Count:  62998
revenue outliers Per:  4.6745483538018275


### 4.7.3 Revenue Outlier Decision

The IQR method identified 62,998 observations (4.67% of the dataset) as statistical upper outliers in `revenue`.

These observations will not be removed.

The `revenue` variable represents a financial characteristic, and extreme values may represent legitimate high-income borrowers rather than data errors.

Therefore, the observations will be retained during Data Quality.

If the strong right-skew of `revenue` affects statistical analysis or machine-learning performance, an appropriate transformation may be considered during later phases.

## 4.8 Outlier Analysis — loan_amnt

The `loan_amnt` feature represents the amount of the loan requested by the borrower.

Its distribution will be examined to determine whether extreme loan amounts are statistically unusual or potentially invalid.

In [22]:
df['loan_amnt'].describe()

count    1.347681e+06
mean     1.440823e+04
std      8.715347e+03
min      5.000000e+02
25%      7.975000e+03
50%      1.200000e+04
75%      2.000000e+04
max      4.000000e+04
Name: loan_amnt, dtype: float64

In [23]:
Q1_loan = df["loan_amnt"].quantile(0.25)
Q3_loan = df["loan_amnt"].quantile(0.75)

IQR_loan = Q3_loan - Q1_loan

upper_bound_loan = Q3_loan + 1.5 * IQR_loan

print("Q1:", Q1_loan)
print("Q3:", Q3_loan)
print("IQR:", IQR_loan)
print("Upper Bound:", upper_bound_loan)

Q1: 7975.0
Q3: 20000.0
IQR: 12025.0
Upper Bound: 38037.5


In [24]:
loan_outliers = df[df["loan_amnt"] > upper_bound_loan]

print("Loan amount outlier count:", len(loan_outliers))
print("Loan amount outlier percentage:", (len(loan_outliers) / len(df)) * 100)

Loan amount outlier count: 7131
Loan amount outlier percentage: 0.5291311519565832


### 4.8.1 Loan Amount Outlier Decision

The IQR method identified 7,131 observations (0.53% of the dataset) as statistical upper outliers in `loan_amnt`.

The maximum loan amount is 40,000, which remains a plausible loan value.

Therefore, these observations will be retained and will not be removed during Data Quality.

Statistical extremeness alone is not sufficient justification for removing valid loan records.

## 4.9 Validity Analysis — fico_n

The `fico_n` feature represents the borrower's FICO credit score.

Because credit scores have a meaningful numerical range, the feature will be examined for unusually low or high values and potential validity issues.

In [25]:
df['fico_n'].describe()

count    1.347681e+06
mean     6.981589e+02
std      3.184872e+01
min      6.120000e+02
25%      6.720000e+02
50%      6.920000e+02
75%      7.120000e+02
max      8.475000e+02
Name: fico_n, dtype: float64

In [26]:
Q1_fico = df["fico_n"].quantile(0.25)
Q3_fico = df["fico_n"].quantile(0.75)

IQR_fico = Q3_fico - Q1_fico

lower_bound_fico = Q1_fico - 1.5 * IQR_fico
upper_bound_fico = Q3_fico + 1.5 * IQR_fico

print("Q1:", Q1_fico)
print("Q3:", Q3_fico)
print("IQR:", IQR_fico)
print("Lower Bound:", lower_bound_fico)
print("Upper Bound:", upper_bound_fico)

Q1: 672.0
Q3: 712.0
IQR: 40.0
Lower Bound: 612.0
Upper Bound: 772.0


In [27]:
fico_outliers = df[df["fico_n"] > upper_bound_fico]

print("FICO upper outlier count:", len(fico_outliers))
print("FICO upper outlier percentage:", (len(fico_outliers) / len(df)) * 100)


FICO upper outlier count: 46490
FICO upper outlier percentage: 3.44962940042933


### 4.9.1 FICO Outlier Decision

The IQR method identified 46,490 observations (3.45% of the dataset) as statistical upper outliers in `fico_n`.

The upper IQR boundary is 772, while the maximum observed value is 847.5.

These observations will be retained because high FICO scores can represent valid borrowers with strong credit profiles.

Therefore, statistical outlier status alone is not considered sufficient justification for removing these records.

In [29]:
df['emp_length'].value_counts(dropna=False)

emp_length
10+ years    442667
2 years      122091
< 1 year     108517
3 years      107862
1 year        88841
5 years       84326
4 years       80761
NI            78188
6 years       62877
8 years       60808
7 years       59724
9 years       51019
Name: count, dtype: int64

### 4.10.1 Decision — `emp_length`

The `emp_length` column contains consistent categorical values representing employment length.

The observed categories include values from `< 1 year` through `10+ years`, along with the `NI` category.

No obvious formatting inconsistencies or duplicate category labels were identified.

The `NI` category will be retained as a separate category because its meaning has not been established as a missing-value indicator.

No transformation is required for `emp_length` during Data Quality.

In [30]:
df['home_ownership_n'].value_counts(dropna=False)

home_ownership_n
MORTGAGE    666594
RENT        535585
OWN         144985
OTHER          517
Name: count, dtype: int64

### 4.10.2 Decision — `home_ownership_n`

The `home_ownership_n` column contains four consistent categorical values:

- `MORTGAGE`
- `RENT`
- `OWN`
- `OTHER`

No missing values or obvious formatting inconsistencies were identified.

The `OTHER` category contains relatively few observations but represents a valid category in the dataset.

Therefore, `home_ownership_n` will be retained without modification.

In [31]:
df['purpose'].value_counts(dropna=False)

purpose
debt_consolidation    781206
credit_card           295551
home_improvement       87684
other                  78263
major_purchase         29543
medical                15606
small_business         15575
car                    14647
moving                  9520
vacation                9083
house                   7294
wedding                 2350
renewable_energy         936
educational              423
Name: count, dtype: int64

### 4.10.3 Decision — `purpose`

The `purpose` column contains 14 distinct loan-purpose categories.

The category labels are consistently formatted, and no obvious duplicate or inconsistent labels were identified.

No missing values are present.

Although the distribution is highly concentrated in a few categories, this represents the underlying distribution of loan purposes rather than a data-quality issue.

Therefore, `purpose` will be retained without modification.


In [32]:
df['addr_state'].value_counts(dropna=False)

addr_state
CA    196805
TX    110307
NY    110075
FL     95815
IL     51858
NJ     48576
PA     45640
OH     43935
GA     43470
VA     38107
NC     37812
MI     35302
AZ     32733
MD     31295
MA     31066
CO     29722
WA     29218
MN     24001
IN     21720
MO     21328
TN     20393
NV     20282
CT     19787
WI     17782
AL     16640
OR     16415
SC     16004
LA     15520
KY     12868
OK     12291
KS     11262
AR     10058
UT     10045
NM      7374
HI      6763
MS      6593
NH      6463
RI      5879
WV      4888
MT      3831
DE      3804
NE      3591
DC      3485
AK      3194
WY      2926
SD      2770
VT      2655
ME      2029
ID      1689
ND      1601
IA        14
Name: count, dtype: int64

### 4.10.4 Decision — `addr_state`

The `addr_state` column contains two-letter state abbreviations with a consistent format.

No obvious formatting inconsistencies, duplicate representations, or missing values were identified.

Some states have substantially fewer observations than others, such as `IA`, but low frequency alone does not indicate a data-quality problem.

Therefore, `addr_state` will be retained without modification.

In [33]:
df['zip_code'].value_counts(dropna=False).head(10)

zip_code
945xx    15013
750xx    14496
112xx    13848
606xx    12477
300xx    12164
331xx    11352
070xx    10786
100xx    10653
891xx    10636
900xx    10588
Name: count, dtype: int64

### 4.10.5 Decision — `zip_code`

The `zip_code` column contains consistently formatted three-character ZIP code prefixes, such as `945xx`, `750xx`, and `112xx`.

No obvious formatting inconsistencies were identified in the inspected values.

The single missing ZIP code will be represented as `Unknown`, as previously decided.

No other transformation will be applied to `zip_code`.

In [34]:
df['issue_d'].describe()

count        1347681
unique           139
top       2016-03-01
freq           48927
Name: issue_d, dtype: object

In [35]:
pd.to_datetime(df['issue_d'],errors='coerce').isna().sum()

np.int64(0)

### 4.11 Valid — `issue_d`

All 1,347,681 values in `issue_d` were successfully converted to valid dates during validation.

No invalid or unparseable date values were identified.

The `issue_d` column will therefore be retained without modification.

The column may be explicitly converted to datetime again when loading the final analytical dataset.


## 4.12 Apply Approved Data-Quality Decisions

Based on the investigation performed in this phase, the following controlled cleaning decisions will be applied:

- Exclude `desc` because approximately 91.16% of its values are missing.
- Replace missing `title` values with `Unknown`.
- Replace the single missing `zip_code` value with `Unknown`.
- Convert the anomalous `dti_n = 999` values to `NaN`.
- Retain statistical outliers in `revenue`, `loan_amnt`, and `fico_n` because they may represent valid observations.
- Retain valid categorical values without modification.
- Retain `issue_d` because no invalid dates were identified.

The original processed dataset will remain unchanged. A separate analytical dataset will be created for downstream analysis.

In [37]:
df_clean = df.copy()

# Removing high missingness free-text column
df_clean = df_clean.drop(columns=['desc'])

# Handling missing categorical values
df_clean['title'] = df_clean['title'].fillna('Unknown')
df_clean['zip_code'] = df_clean['zip_code'].fillna('Unknown')

# handling anomalous DTI placeholder
df_clean.loc[df_clean['dti_n'] == 999, 'dti_n'] = pd.NA

print('Orinal dataset shape: ', df.shape)
print('Cleaned dataset shape: ', df_clean.shape)

Orinal dataset shape:  (1347681, 15)
Cleaned dataset shape:  (1347681, 14)


In [38]:
print("Remaining missing title:", df_clean["title"].isna().sum())
print("Remaining missing zip_code:", df_clean["zip_code"].isna().sum())
print("Remaining dti_n = 999:", (df_clean["dti_n"] == 999).sum())
print("desc column present:", "desc" in df_clean.columns)

Remaining missing title: 0
Remaining missing zip_code: 0
Remaining dti_n = 999: 0
desc column present: False


In [39]:
from pathlib import Path

cleaned_path = Path("../data/processed/LC_loans_granting_model_dataset_cleaned.csv")

df_clean.to_csv(cleaned_path, index=False)

print("Cleaned dataset saved to:", cleaned_path)
print("\nData quality checks completed.")
print("\nCleaned dataset shape: ", df_clean.shape)

Cleaned dataset saved to: ..\data\processed\LC_loans_granting_model_dataset_cleaned.csv

Data quality checks completed.

Cleaned dataset shape:  (1347681, 14)


# Phase 4 — Data Quality Conclusion

The data-quality assessment identified and addressed the major quality issues in the processed loan dataset.

## Cleaning Performed

- Excluded `desc` because approximately 91.16% of its values were missing.
- Replaced missing `title` values with `Unknown`.
- Replaced the single missing `zip_code` value with `Unknown`.
- Converted the 38 anomalous `dti_n = 999` values to missing (`NaN`).
- Retained statistical outliers in `revenue`, `loan_amnt`, and `fico_n` because they may represent valid observations.
- Retained valid categorical values without modification.
- Validated `issue_d` and found no invalid dates.

## Final Dataset

- Original rows: 1,347,681
- Final rows: 1,347,681
- Original columns: 15
- Final columns: 14
- Loan records removed: 0

The cleaned dataset is saved as:

`data/processed/LC_loans_granting_model_dataset_cleaned.csv`

The dataset is now ready for downstream business analysis.